In [46]:
import pandas as pd
import numpy as np
from libpysal.weights import full2W
from esda.moran import Moran
from pyprojroot import here

In [47]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results"

In [48]:
crime = pd.read_csv(INTERIM_DATA_DIR/"crime_count_2015-2025.csv", encoding='latin-1')
adj = pd.read_csv(INTERIM_DATA_DIR/'adj_combined.csv', encoding='latin-1', index_col=0, dtype=str)

In [49]:
crime["CVEGEO"] = crime["CVEGEO"].astype(str).str.zfill(5)
adj.index = adj.index.astype(str).str.zfill(5)
adj.columns = adj.columns.astype(str).str.zfill(5)

In [50]:
common_ids = sorted(set(adj.index) & set(crime["CVEGEO"]))

In [51]:
adj = adj.loc[common_ids, common_ids]

In [52]:
w = full2W(adj.values.astype(int), ids=common_ids)
w.transform = "r"  # row-standardize

print(f"True islands: {w.islands}")  # expect just ['06009']

True islands: []


In [53]:
results = []
years = sorted(crime["year"].unique())

for year in years:
    yr_data = crime[crime["year"] == year].set_index("CVEGEO").reindex(common_ids)

    if yr_data["crime_count"].isna().any():
        missing = yr_data[yr_data["crime_count"].isna()].index.tolist()
        print(f"Year {year}: missing crime_count for {len(missing)} municipalities — check data")
        continue

    y = yr_data["crime_count"].values.astype(float)
    mi = Moran(y, w, permutations=999)

    results.append({
        "year": year,
        "morans_I": mi.I,
        "expected_I": mi.EI,
        "p_sim": mi.p_sim,
        "p_norm": mi.p_norm,
        "z_sim": mi.z_sim,
    })

moran_results = pd.DataFrame(results)
print(moran_results)

    year  morans_I  expected_I  p_sim        p_norm      z_sim
0   2015  0.199222   -0.000404  0.001  1.140717e-60  15.513147
1   2016  0.160096   -0.000404  0.001  7.596689e-40  16.168011
2   2017  0.100697   -0.000404  0.004  8.671960e-17   8.672638
3   2018  0.127712   -0.000404  0.001  5.338202e-26  12.199149
4   2019  0.112037   -0.000404  0.001  2.140992e-20  11.726553
5   2020  0.049977   -0.000404  0.007  3.370586e-05   5.280175
6   2021  0.035666   -0.000404  0.006  2.988415e-03   4.962962
7   2022  0.049717   -0.000404  0.004  3.700192e-05   6.039339
8   2023  0.080655   -0.000404  0.001  2.522405e-11  10.260270
9   2024  0.075860   -0.000404  0.002  3.444794e-10   9.519241
10  2025  0.091539   -0.000404  0.001  3.792447e-14  11.393908


In [54]:
moran_results.to_csv(RESULTS_DATA_DIR/"esda/yearly_morans_I.csv", index=False)